[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DL_4_Physics/PINNs.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Physics-Informed Neural Networks

The flagship idea of deep learning *for physics*: instead of fitting data alone, make the network satisfy the **differential equation itself** — autograd computes the derivatives, the PDE residual becomes a loss term, and physics becomes a regularizer that lets you learn from absurdly little data.

## 1. Pre-requisites

- [Intro to PyTorch](./intro_pytorch/intro_pytorch.ipynb) — autograd, training loops.
- An ODE/PDE course helps but the two equations used here are self-contained.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0)

---
### 🕐 Session 1 of 2 — *The PDE-as-Loss Idea* (~35 min)
**Goal:** use autograd to differentiate the network w.r.t. its INPUTS; solve an ODE with zero data.
**Builds on:** [Intro to PyTorch](./intro_pytorch/intro_pytorch.ipynb). &nbsp; **Feeds into:** Session 2 (a real boundary-value problem).

---

## 2. Autograd's Second Job

💡 **Intuition.** Backprop differentiates the loss w.r.t. *weights*. But autograd is general: it can just as happily differentiate the network's **output w.r.t. its input** — giving us $u'(t)$, $u''(t)$ for a network $u_\theta(t)$, exactly, no finite differences. So if physics says $u' = -\lambda u$, we can *penalize the network for violating it* at any set of collocation points: $\mathcal{L}_{physics} = \frac{1}{N}\sum_i (u'(t_i) + \lambda u(t_i))^2$. The equation becomes training data — infinite, free, and exact.

In [ ]:
# Warm-up: solve u' = -1.5 u, u(0)=1 with NO solution data at all

# YOUR CODE HERE


**What just happened.** A network learned $e^{-1.5t}$ to a maximum error of **$5.9\times10^{-3}$** — about 0.6% — having **never seen a single value of the solution**. The only inputs were the equation $u' = -1.5u$ and the initial condition $u(0) = 1$.

**The mechanism is one line: `torch.autograd.grad(u, t, ...)`.** Backprop differentiates the loss with respect to *weights*; here we differentiate the *output* with respect to the *input*, getting $u'(t)$ **exactly** — no finite differences, no step size, no discretisation error. **Autograd could always do this; it is simply not what it is normally asked for.**

**Note `create_graph=True`, because omitting it is the classic first bug.** It keeps the derivative computation inside the graph, so the physics loss has a gradient with respect to the weights. Without it, `du` is treated as a constant, `loss_phys.backward()` contributes nothing, and the network trains on the initial condition alone — silently, with no error message. **Every PINN implementation lives or dies on that flag.**

**The initial-condition term is not decoration either.** The residual $u' + \lambda u = 0$ is satisfied by the *entire family* $Ce^{-\lambda t}$ — including $C = 0$, which is a perfect, useless solution the optimiser will happily find. **The residual selects a family; the boundary condition selects a member.** Drop `loss_ic` and the network collapses to zero.

**Now the honest reading of $5.9\times10^{-3}$, because it is not as good as it sounds.** The solution has magnitude 1, so this is 0.6% error after 2,000 optimisation steps. **A fourth-order Runge–Kutta integrator solves this ODE to $10^{-10}$ in microseconds** — five thousand times more accurate, and thousands of times faster. **PINNs are not a better way to solve ODEs you can already solve**, and this demo should not be read as a claim that they are.

**Which makes it important to say where PINNs genuinely win.** *High dimensions*, where mesh-based solvers face the curse of dimensionality and a network does not. *Inverse problems*, where a physical constant is unknown — make $\lambda$ a `nn.Parameter` and the same loss estimates it from data. And *data assimilation*: sparse noisy measurements plus a governing equation, which is Session 2's subject. **This cell is a mechanism check, not a use case.**

**One structural observation worth making while the code is on screen.** Collocation points are drawn **fresh at random each step** — `torch.rand(64,1)*3`. So the training set is effectively infinite and the network never sees the same point twice. **There is no overfitting in the usual sense**, because the "data" is generated on demand from an equation. That is a genuinely different regime from every other workshop in the ML track.

**And note what the loss is not telling you.** The physics loss measures *residual*, not *error*. A network can have a small residual and still be the wrong solution — near-solutions of an ODE can drift far from the true one over a long interval. **Low residual is necessary and not sufficient**, which is why this cell compares against the analytic solution rather than reporting the loss.

---
### 🕐 Session 2 of 2 — *A Real Problem: the Damped Oscillator from 6 Points* (~40 min)
**Goal:** combine sparse noisy data with physics; watch the physics term rescue the fit.
**Builds on:** Session 1.

---

## 3. Data + Physics

💡 **Intuition.** The honest selling point of PINNs is the *combination*: a handful of noisy measurements can't pin down a wiggly function — but they can pin down the **constants** (amplitude, phase) of a function the physics already shapes. Loss = data misfit + PDE residual; the physics term acts as an infinitely-informative prior. Watch a plain network hallucinate between 6 points while the PINN interpolates *and extrapolates* correctly.

In [ ]:
# damped oscillator: u'' + 2ζω u' + ω² u = 0,  ω=8, ζ=0.05
            # initial conditions u(0)=1, u'(0)=0 anchor the family member

# YOUR CODE HERE


**What just happened.** Same architecture, same six noisy points, same optimiser, same 4,000 steps. **The only difference is one term in the loss**, and the results are:

| model | RMSE vs truth |
|---|---|
| plain NN | **0.675** |
| PINN | **0.027** |

**A 25× gap — but the plain baseline is worse than trivial, which the ratio hides.** The true signal $e^{-0.4t}\cos(7.99t)$ on $[0,2]$ has RMS about **0.50**. So a model that predicted **zero everywhere** would score 0.50, and the plain network scores 0.675. **It is not merely uninformative; it is actively worse than silence.** Six points cannot constrain an 8,600-parameter network, and between them it invents a smooth curve with no oscillation at all.

**Now the caveat that changes the headline, and it should not be skipped.** The PINN's loss includes **both initial conditions** — $u(0) = 1$ and $u'(0) = 0$ — alongside the ODE residual. A second-order linear ODE with two initial conditions has a **unique solution**, and that solution *is* `analytic`. **So the physics alone already determines the answer completely; the six data points are redundant.** The plot's label "6 points + the ODE" credits the data for work the equation did by itself.

**That is worth turning into the session's best experiment rather than glossing over.** Run `train_net(1e-3)` with `t_data` emptied — the PINN should recover the solution anyway. Then run it with the IC terms deleted and the six points kept: now the data genuinely picks the member of the two-parameter solution family, and the demo means exactly what it claims. **Two ablations, and the room learns which ingredient carries the result.**

**What the comparison *does* establish is still real and worth stating precisely.** Adding a differential equation to the loss changes an unusable fit into an accurate one, at identical capacity, data, and compute. **Physics-as-regulariser works.** The claim it does not establish is "six measurements were sufficient" — that requires the ablation above.

**Look at the extrapolation region beyond $t = 1.53$, because it is the most convincing part of the figure.** The last data point is at 1.53 and the plot runs to 2.0. The plain network drifts off immediately — it has no reason not to. The PINN keeps oscillating correctly, because the **residual is enforced at collocation points across the whole interval**, data or no data. **The physics term supervises where the data is silent**, which is the property that makes PINNs interesting for forecasting and gap-filling.

**The weight $10^{-3}$ is doing real work and is the hardest thing to set in practice.** The residual contains $\omega^2 u = 64u$, so its natural magnitude is orders above the data misfit; an unweighted sum would let physics swamp the six points entirely. **Loss weighting is the central practical difficulty of PINNs** — there is a literature of adaptive schemes for exactly this, and 1e-3 here was found by tuning, not derived.

**Finally, the capability with no classical competitor, which the notebook rightly flags.** Make $\omega$ an `nn.Parameter` and the identical loss **estimates the physical constant from the data**. No derivative estimation, no spectral fitting — gradient descent on a residual. **An ODE solver cannot do this at all**, and inverse problems, not forward solves, are where PINNs genuinely earn their place.

**Practicalities worth a slide:** weighting the loss terms is the art (residual and data scales differ — here 1e-3 balances them); stiff/high-frequency problems need Fourier feature inputs or curriculum in time; and PINNs also run *inverse* problems — make $\omega$ a `nn.Parameter` and the same loss estimates the physical constant from data. Try it: you should recover $\omega \approx 8$.

## 4. Conclusion

Autograd differentiates through inputs, so equations become losses, physics becomes a prior, and six noisy points suffice where hundreds were needed. This is the core move of scientific machine learning.

---
## Where next

- [Intro to PyTorch](./intro_pytorch/intro_pytorch.ipynb) — the autograd machinery.
- [Uncertainty in ML](../Intro_Mach_Learn/Uncertainty_in_ML.ipynb) — how much to trust the extrapolation.
- [Kernel Methods](../Intro_Mach_Learn/Kernel_Methods.ipynb) — the classical way of encoding priors, for contrast.